# Modélisation : effet des médailles olympiques sur les licenciés

## Objectif
On cherche à mesurer si les médailles obtenues aux JO (par sport) sont associées à une augmentation du nombre de licenciés les années suivantes.

On utilise deux approches complémentaires :

1) **Modèle économétrique (principal)** : estimation de l’effet moyen des médailles sur la **croissance** des licenciés, avec **effets fixes sport** et **effets fixes année**, et erreurs robustes **clusterisées par sport**.

2) **Diagnostics complémentaires (“preuves”)** : modèles prédictifs simples pour quantifier le rôle de l’inertie (niveau passé) vs les médailles.
Attention: Ces diagnostics sont descriptifs/prédictifs, pas causaux.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Package maison
from model.feature_engineering import build_lic_features, merge_medals
from model.modele_econo import run_econo_model


## Construction du panel final sport–année

Nous construisons un panel au niveau *(sport, année)* contenant :
- `nb_licencies` : nombre total de licenciés du sport l’année donnée ;
- des variables descriptives agrégées (sexe, âge, dispersion géographique) ;
- `jo_ref` : JO de référence (dernier JO avant l’année) ;
- `total_medailles` : nombre de médailles obtenues lors du JO de référence pour ce sport (variable clé).


In [ ]:
df_features = build_lic_features(df_lic)
df_model_full = merge_medals(df_features, df_med)

print("df_model_full shape:", df_model_full.shape)
df_model_full.head()


## Sanity checks (cohérence et reproductibilité)

On vérifie :
- unicité des observations (sport–année) ;
- cohérence de l’assignation `jo_ref` ;
- cohérence de `total_medailles` (constante par sport pour un JO donné).


In [ ]:
print("Duplicats (code_sport, annee):", df_model_full.duplicated(["code_sport","annee"]).sum())

# total_medailles doit être constant au sein de (sport, jo_ref)
tmp = df_model_full.groupby(["code_sport","jo_ref"])["total_medailles"].nunique().reset_index()
print("Max nunique(total_medailles) par (sport, jo_ref):", tmp["total_medailles"].max())

# répartition jo_ref par année
df_model_full.groupby(["annee","jo_ref"]).size().head(20)


## Modèle économétrique principal : croissance post-JO (panel avec effets fixes)

Nous estimons :

\[
\Delta \log(1+Lic_{s,t}) = \beta \cdot Med_{s,JO} + \alpha_s + \gamma_t + \varepsilon_{s,t}
\]

- Variable dépendante : \(\Delta \log(1+Lic_{s,t})\) ≈ taux de croissance annuel du nombre de licenciés (robuste aux niveaux très différents entre sports).
- \(\alpha_s\) : **effets fixes sport** (popularité structurelle, culture de pratique, taille de base).
- \(\gamma_t\) : **effets fixes année** (chocs communs : Covid, politiques publiques, tendances globales).
- Erreurs standards **clusterisées par sport** : autorise la corrélation intra-sport dans le temps.

Interprétation : si \(\beta\) est petit, \(100\beta\) ≈ variation en points de % de la croissance annuelle associée à **+1 médaille**.


In [ ]:
results_econo = run_econo_model(
    df_model_full=df_model_full,
    medals_col="total_medailles",
    start_year=2017,
    end_year=2024,
    controls=["part_femmes", "age_mean", "nb_departements_actifs"],  # optionnel
)

print(results_econo.summary())


In [ ]:
beta = float(results_econo.params["med_last"])
ci = results_econo.conf_int().loc["med_last"]

print(f"beta(médailles) = {beta:.6f}")
print(f"Interprétation: +1 médaille -> ~ {100*beta:.2f}% de croissance annuelle (approx.)")
print(f"IC 95%: [{100*ci[0]:.2f}%, {100*ci[1]:.2f}%]")


### Discussion économétrique (limites)

Même avec effets fixes sport et année, l’interprétation causale doit rester prudente :
- les médailles peuvent être endogènes (sports structurellement forts / mieux financés → plus de médailles et plus de licenciés) ;
- effets hétérogènes selon les sports (médiatisation, accessibilité de la pratique, structure fédérale) ;
- le modèle mesure un effet moyen à court terme, pas une trajectoire complète.

Ces limites motivent les diagnostics ci-dessous, qui mettent en évidence le rôle central de l’inertie dans les licenciés.


## Diagnostics / “preuves” complémentaires

Ces diagnostics ne visent pas à établir une causalité stricte, mais à :
1) quantifier le rôle de l’inertie vs médailles en **prédiction hors échantillon** ;
2) illustrer que la dynamique des licenciés est très persistante ;
3) vérifier si les médailles apportent une information marginale dans le cadre du modèle économétrique.


### Preuve 1 — Ablation prédictive (inertie vs médailles)

On compare trois modèles prédictifs simples sur la période test (2022–2024) :
- **M0 : inertie seule** (`log(lic_{t-1})`)
- **M1 : médailles seules**
- **M2 : inertie + médailles**

Attention: Un bon score des “médailles seules” peut refléter une corrélation avec la popularité structurelle (pas causal).
L’enjeu principal est de comparer M0 vs M2 : l’ajout des médailles apporte-t-il un gain prédictif notable ?


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

def eval_ablation_models(df_model_full, train_years=(2017, 2020), test_years=(2022, 2024),
                         medals_col="total_medailles", alpha=1.0):
    df = df_model_full.sort_values(["code_sport","annee"]).copy()
    df["y"] = np.log1p(df["nb_licencies"])
    df["log_lag1"] = df.groupby("code_sport")["y"].shift(1)
    df["trend"] = df["annee"] - df["annee"].min()
    df["med_last"] = df[medals_col]
    df = df.dropna(subset=["log_lag1"]).copy()

    train_df = df[df["annee"].between(*train_years)].copy()
    test_df  = df[df["annee"].between(*test_years)].copy()

    def fit_predict(feats):
        Xtr = pd.get_dummies(train_df[["code_sport"] + feats], columns=["code_sport"], drop_first=True).astype("float32")
        Xte = pd.get_dummies(test_df[["code_sport"] + feats], columns=["code_sport"], drop_first=True).astype("float32")
        Xte = Xte.reindex(columns=Xtr.columns, fill_value=0).astype("float32")

        ytr = train_df["y"].astype("float32").values
        yte = test_df["y"].astype("float32").values

        m = Ridge(alpha=alpha)
        m.fit(Xtr, ytr)
        pred = m.predict(Xte)

        return float(r2_score(yte, pred)), float(mean_absolute_error(np.expm1(yte), np.expm1(pred)))

    rows = []
    for name, feats in [
        ("M0: inertie seule", ["log_lag1"]),
        ("M1: médailles seules", ["med_last"]),
        ("M2: inertie + médailles", ["log_lag1","med_last"]),
    ]:
        r2, mae = fit_predict(feats)
        rows.append({"modele": name, "R2_log": r2, "MAE_niveau": mae})
    return pd.DataFrame(rows)

ablation = eval_ablation_models(df_model_full)
ablation


**Lecture** :
- Si M0 est déjà très performant, cela indique que les licenciés sont fortement **inertiels**.
- Si M2 n’améliore que marginalement M0, les médailles apportent peu d’information prédictive additionnelle dans ce cadre.
- La performance de M1 doit être interprétée prudemment (corrélation structurelle possible).


### Preuve 2 — Poids de l’inertie (coefficients Ridge)

On estime un Ridge sur le train et on examine les coefficients (en valeur absolue).
Si `log_lag1` domine nettement les autres, cela quantifie le rôle central de l’inertie.


In [ ]:
def ridge_coefficients(df_model_full, train_years=(2017, 2020), medals_col="total_medailles", alpha=1.0):
    df = df_model_full.sort_values(["code_sport","annee"]).copy()
    df["y"] = np.log1p(df["nb_licencies"])
    df["log_lag1"] = df.groupby("code_sport")["y"].shift(1)
    df["trend"] = df["annee"] - df["annee"].min()
    df["med_last"] = df[medals_col]
    df = df.dropna(subset=["log_lag1"]).copy()

    train_df = df[df["annee"].between(*train_years)].copy()
    feats = ["log_lag1", "trend", "med_last"]

    X = pd.get_dummies(train_df[["code_sport"] + feats], columns=["code_sport"], drop_first=True).astype("float32")
    y = train_df["y"].astype("float32").values

    m = Ridge(alpha=alpha)
    m.fit(X, y)

    coef = pd.Series(m.coef_, index=X.columns).sort_values(key=lambda s: s.abs(), ascending=False)
    return coef

coef = ridge_coefficients(df_model_full)
coef.head(15)


**Lecture** :
- un coefficient très élevé sur `log_lag1` reflète l’inertie : le niveau passé explique la majeure partie du niveau actuel.
- les médailles ont généralement un poids plus faible dans la prédiction du niveau.


### Preuve 3 — Les médailles apportent-elles une information marginale dans le modèle éco ?

On compare :
- un modèle **sans** médailles (FE sport + FE année + contrôles)
- un modèle **avec** médailles

On réalise un test de Wald sur l’hypothèse nulle \(H_0 : \beta = 0\) (coefficient médailles nul).


In [ ]:
import statsmodels.formula.api as smf

def test_medals_incremental(df_model_full, medals_col="total_medailles", start=2017, end=2024):
    dfp = df_model_full.sort_values(["code_sport","annee"]).copy()
    dfp["log_lic"] = np.log1p(dfp["nb_licencies"])
    dfp["dlog_lic"] = dfp.groupby("code_sport")["log_lic"].diff(1)
    dfp = dfp[dfp["annee"].between(start, end)].dropna(subset=["dlog_lic"]).copy()
    dfp["med_last"] = dfp[medals_col]

    controls = [c for c in ["part_femmes","age_mean","nb_departements_actifs"] if c in dfp.columns]
    ctrl = (" + " + " + ".join(controls)) if controls else ""

    f0 = f"dlog_lic ~ C(code_sport) + C(annee){ctrl}"
    f1 = f"dlog_lic ~ med_last + C(code_sport) + C(annee){ctrl}"

    m0 = smf.ols(f0, data=dfp).fit(cov_type="cluster", cov_kwds={"groups": dfp["code_sport"]})
    m1 = smf.ols(f1, data=dfp).fit(cov_type="cluster", cov_kwds={"groups": dfp["code_sport"]})

    wald = m1.wald_test("med_last = 0")
    return m0, m1, wald

m0, m1, wald = test_medals_incremental(df_model_full)
print(wald)
print("beta medals:", m1.params["med_last"])


**Lecture** :
- Si la p-value est faible, l’effet des médailles est statistiquement différent de zéro, une fois contrôlés les effets fixes.
- Cela renforce l’association conditionnelle mesurée, sans garantir une causalité parfaite.


## Conclusion (modélisation)

- Le modèle économétrique (panel avec effets fixes sport et année, erreurs cluster sport) fournit une estimation interprétable de l’effet moyen des médailles sur la croissance des licenciés.
- Les diagnostics montrent que la dynamique des licenciés est fortement dominée par l’inertie (niveau passé), et que l’information apportée par les médailles est plus marginale.
- La causalité doit être interprétée avec prudence (endogénéité possible), mais la démarche est cohérente et reproductible.
